<a href="https://colab.research.google.com/github/kvanoost/AGP/blob/main/4_cache_for_inference_all_years_drive_mount_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trajectory cache for Model 5 inference

**Purpose**

Build all-year trajectory predictor caches for applying:

`Model 5 — RF-GEE-plus-S2-Temporal-24m-plus-CNNPatchProbs`

This cache notebook avoids NICFI entirely and creates only the predictors needed by Model 5:

1. **GEE annual point embeddings**: `n × 64`
2. **GEE annual p3 patches** for CNN-GEE: `n × 3 × 3 × 64`
3. **Sentinel-2 temporal T24**: `n × 24 × 11`

The logic is designed to be consistent with the model-training cache notebook: annual predictors are aligned to the target year, and S2 temporal predictors use **Jan Y−1 to Dec Y**.


## 1. User settings

In [ ]:
# ============================================================
# 1. USER SETTINGS
# ============================================================

import os
import re
import json
import time
import math
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Google Earth Engine project
# ------------------------------------------------------------
EE_PROJECT = "panafricanlu"   # change only if needed

# ------------------------------------------------------------
# Google Drive / project location
# ------------------------------------------------------------
# IMPORTANT:
# Do not create /content/drive/MyDrive/... folders before Google Drive is mounted.
# This notebook now defines only the MyDrive-relative project folder here.
# Absolute /content/drive paths are constructed only after the mount in Block 2.

DRIVE_MOUNT_POINT = "/content/drive"
PROJECT_ROOT_IN_MYDRIVE = "Colab Notebooks/PanAfrica_LU/NN"

# Input files, relative to PROJECT_ROOT
REFERENCE_CSV_NAME = "Kasai_reference.csv"
MASTER_SPLIT_REL_PATH = os.path.join("Models", "model_master_split_train_val_test.csv")

# ------------------------------------------------------------
# Output destination, relative to DATA_DIR
# ------------------------------------------------------------
DESTINATION_TRAJ_CACHE_REL_DIR = os.path.join(
    "Traj_cache",
    "all_zones_2017_2025"
)

# ------------------------------------------------------------
# Trajectory settings
# ------------------------------------------------------------
TRAJ_START_YEAR = 2017
TRAJ_END_YEAR = 2025
TRAJ_YEARS = list(range(TRAJ_START_YEAR, TRAJ_END_YEAR + 1))

TARGET_ZONE_FOR_TRAJ_CACHE = "all"   # "all", None, or one of: Ilebo, Mbuji, Mushie, Tshikapa

# ------------------------------------------------------------
# Overwrite / resume controls
# ------------------------------------------------------------
OVERWRITE_MASTER = False

OVERWRITE_GEE_POINT_FINAL = False
OVERWRITE_GEE_POINT_CHUNKS = False

OVERWRITE_GEE_PATCH_FINAL = False
OVERWRITE_GEE_PATCH_CHUNKS = False

OVERWRITE_S2_FINAL = True
OVERWRITE_S2_CHUNKS = False

# ------------------------------------------------------------
# Earth Engine extraction settings
# ------------------------------------------------------------
EE_BATCH_SIZE_GEE_POINT = 200
EE_BATCH_SIZE_GEE_PATCH = 100
EE_BATCH_SIZE_S2 = 1          # keep modest; S2 temporal is memory-heavy

EE_SLEEP_SECONDS = 1.0
EE_TILE_SCALE_GEE = 4
EE_TILE_SCALE_S2 = 8

# ------------------------------------------------------------
# GEE embedding settings
# ------------------------------------------------------------
GEE_EMBEDDING_COLLECTION = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
GEE_SCALE = 10
GEE_PATCH_SIZE = 3
GEE_N_BANDS = 64

# ------------------------------------------------------------
# Sentinel-2 T24 settings
# ------------------------------------------------------------
S2_TEMPORAL_LENGTH = 24
S2_TEMPORAL_MODE = "prev_plus_target_year"  # Jan Y-1 ... Dec Y

S2_SR_IC = "COPERNICUS/S2_SR_HARMONIZED"
S2_CLOUD_PROB_IC = "COPERNICUS/S2_CLOUD_PROBABILITY"

S2_SCALE = 10
S2_CLOUD_PROB_THRESHOLD = 40
S2_CLOUDY_PIXEL_PERCENTAGE_MAX = 85

S2_RAW_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
S2_INDEX_BANDS = ["NDVI", "EVI", "NDWI", "NBR"]
S2_FEATURE_BANDS = S2_RAW_BANDS + S2_INDEX_BANDS + ["valid_obs_count"]

# Important: sampleRegions can drop rows if bands are masked.
# Use explicit nodata in Earth Engine, convert back to NaN in Python for spectral/index bands.
S2_NODATA_VALUE = -9999.0

print("User settings loaded.")
print("Drive mount point:", DRIVE_MOUNT_POINT)
print("Project folder in MyDrive:", PROJECT_ROOT_IN_MYDRIVE)
print("Trajectory years:", TRAJ_START_YEAR, "to", TRAJ_END_YEAR)


User settings loaded.
Drive mount point: /content/drive
Project folder in MyDrive: Colab Notebooks/PanAfrica_LU/NN
Trajectory years: 2017 to 2025


## 2. Mount Google Drive and initialize Earth Engine

In [ ]:
# ============================================================
# 2. GOOGLE DRIVE + EARTH ENGINE INITIALIZATION
# ============================================================

import os
import shutil
from pathlib import Path
from datetime import datetime

from google.colab import drive

# ------------------------------------------------------------
# 2.1 Mount Google Drive safely at the standard Colab path
# ------------------------------------------------------------
# What went wrong before:
# Block 1 created folders under /content/drive/MyDrive before Drive was mounted.
# That creates a local fake /content/drive folder. Then drive.mount('/content/drive')
# fails with: "Mountpoint must not already contain files".
#
# This fixed block never creates Drive folders before mounting. If a fake local
# /content/drive already exists, it is moved safely to /tmp instead of deleted.

DRIVE_MOUNT_POINT = "/content/drive"
LOCAL_FAKE_DRIVE_BACKUP = None

# If /content/drive exists and is non-empty before mounting, it is usually a fake local folder.
# Move it to /tmp so real Google Drive can mount cleanly.
if os.path.exists(DRIVE_MOUNT_POINT):
    existing_items = os.listdir(DRIVE_MOUNT_POINT)
else:
    existing_items = []

looks_like_real_mounted_drive = os.path.exists(os.path.join(DRIVE_MOUNT_POINT, "MyDrive"))

if existing_items and not looks_like_real_mounted_drive:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    LOCAL_FAKE_DRIVE_BACKUP = f"/tmp/local_fake_content_drive_backup_{stamp}"
    print("WARNING: found a non-empty local /content/drive before mounting Google Drive.")
    print("This is probably the fake-drive folder created by earlier runs.")
    print("Moving it safely to:")
    print(LOCAL_FAKE_DRIVE_BACKUP)
    shutil.move(DRIVE_MOUNT_POINT, LOCAL_FAKE_DRIVE_BACKUP)

# Ensure clean mount point exists.
os.makedirs(DRIVE_MOUNT_POINT, exist_ok=True)

print("Mounting Google Drive at:", DRIVE_MOUNT_POINT)
drive.mount(DRIVE_MOUNT_POINT)

MYDRIVE_ROOT = os.path.join(DRIVE_MOUNT_POINT, "MyDrive")
if not os.path.exists(MYDRIVE_ROOT):
    raise RuntimeError("Google Drive mount succeeded, but MyDrive was not found.")

print("Google Drive mounted correctly.")
print("MYDRIVE_ROOT:", MYDRIVE_ROOT)

# ------------------------------------------------------------
# 2.2 Construct all absolute paths AFTER mounting Drive
# ------------------------------------------------------------

PROJECT_ROOT = os.path.join(MYDRIVE_ROOT, PROJECT_ROOT_IN_MYDRIVE)
DATA_DIR = os.path.join(PROJECT_ROOT, "Data")

REFERENCE_CSV = os.path.join(DATA_DIR, REFERENCE_CSV_NAME)
MASTER_SPLIT_PATH = os.path.join(DATA_DIR, MASTER_SPLIT_REL_PATH)

DESTINATION_TRAJ_CACHE_DIR = os.path.join(DATA_DIR, DESTINATION_TRAJ_CACHE_REL_DIR)

TRAJ_MASTER_CSV = os.path.join(
    DESTINATION_TRAJ_CACHE_DIR,
    f"trajectory_master_all_zones_{TRAJ_START_YEAR}_{TRAJ_END_YEAR}.csv"
)

GEE_POINT_DIR = os.path.join(DESTINATION_TRAJ_CACHE_DIR, "gee_embedding_point")
GEE_PATCH_DIR = os.path.join(DESTINATION_TRAJ_CACHE_DIR, f"gee_embedding_patch_p{GEE_PATCH_SIZE}")
S2_TEMPORAL_DIR = os.path.join(
    DESTINATION_TRAJ_CACHE_DIR,
    f"s2_temporal_T{S2_TEMPORAL_LENGTH}_{S2_TEMPORAL_MODE}"
)

TRAJ_GEE_POINT_NPZ = os.path.join(
    GEE_POINT_DIR,
    f"trajectory_gee_embedding_point_all_zones_{TRAJ_START_YEAR}_{TRAJ_END_YEAR}.npz"
)

TRAJ_GEE_PATCH_NPZ = os.path.join(
    GEE_PATCH_DIR,
    f"trajectory_gee_embedding_patch_p{GEE_PATCH_SIZE}_all_zones_{TRAJ_START_YEAR}_{TRAJ_END_YEAR}.npz"
)

TRAJ_S2_TEMPORAL_NPZ = os.path.join(
    S2_TEMPORAL_DIR,
    f"trajectory_s2_temporal_T{S2_TEMPORAL_LENGTH}_{S2_TEMPORAL_MODE}_all_zones_{TRAJ_START_YEAR}_{TRAJ_END_YEAR}.npz"
)

# Now it is safe to create output folders in the real Google Drive.
for p in [DESTINATION_TRAJ_CACHE_DIR, GEE_POINT_DIR, GEE_PATCH_DIR, S2_TEMPORAL_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("\nResolved paths in real Google Drive:")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("REFERENCE_CSV:", REFERENCE_CSV)
print("MASTER_SPLIT_PATH:", MASTER_SPLIT_PATH)
print("DESTINATION_TRAJ_CACHE_DIR:", DESTINATION_TRAJ_CACHE_DIR)

print("\nOutputs:")
print("TRAJ_MASTER_CSV:", TRAJ_MASTER_CSV)
print("TRAJ_GEE_POINT_NPZ:", TRAJ_GEE_POINT_NPZ)
print("TRAJ_GEE_PATCH_NPZ:", TRAJ_GEE_PATCH_NPZ)
print("TRAJ_S2_TEMPORAL_NPZ:", TRAJ_S2_TEMPORAL_NPZ)

# ------------------------------------------------------------
# 2.3 Optional: show where fake-drive files were moved
# ------------------------------------------------------------
if LOCAL_FAKE_DRIVE_BACKUP is not None:
    print("\nA fake local drive folder was moved to:")
    print(LOCAL_FAKE_DRIVE_BACKUP)
    print("If it contains useful cache chunks, recover them manually from there before resetting the runtime.")

# ------------------------------------------------------------
# 2.4 Initialize Earth Engine
# ------------------------------------------------------------

import ee

try:
    ee.Initialize(project=EE_PROJECT)
    print("\nEarth Engine initialized with project:")
    print(EE_PROJECT)

except Exception:
    print("\nEarth Engine initialization failed.")
    print("Starting authentication...")

    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

    print("\nEarth Engine initialized after authentication with project:")
    print(EE_PROJECT)

print("\nBlock 2 complete.")


Mounting Google Drive at: /content/drive
Mounted at /content/drive
Google Drive mounted correctly.
MYDRIVE_ROOT: /content/drive/MyDrive

Resolved paths in real Google Drive:
PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN
DATA_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data
REFERENCE_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv
MASTER_SPLIT_PATH: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Models/model_master_split_train_val_test.csv
DESTINATION_TRAJ_CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025

Outputs:
TRAJ_MASTER_CSV: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/trajectory_master_all_zones_2017_2025.csv
TRAJ_GEE_POINT_NPZ: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/gee_embedding_point/trajectory_gee_embedding_point_all_zones_2017_2025.npz
TRA

## 3. Helper functions

In [ ]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def clean_id(x):
    return str(x).strip()

def assert_exists(path, label="path"):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} not found:\n{path}")
    print(f"OK - {label}:\n{path}")

def find_first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def detect_lon_lat_columns(df):
    lon_candidates = ["lon", "longitude", "x", "POINT_X", "Long", "LONGITUDE"]
    lat_candidates = ["lat", "latitude", "y", "POINT_Y", "Lat", "LATITUDE"]
    return find_first_existing_column(df, lon_candidates), find_first_existing_column(df, lat_candidates)

def detect_zone_column(df):
    return find_first_existing_column(df, ["zone_id", "zone", "site", "site_id", "region"])

def make_sample_id(unit_id, year, month):
    return f"{str(unit_id)}_{int(year)}_{int(month):02d}"

def make_traj_sample_id(unit_id, year):
    return f"{str(unit_id)}_{int(year)}"

def get_s2_month_starts_for_target_year(target_year):
    target_year = int(target_year)
    starts = []
    for y in [target_year - 1, target_year]:
        for m in range(1, 13):
            starts.append(f"{y}-{m:02d}-01")
    return starts

def get_s2_window_for_target_year(target_year):
    target_year = int(target_year)
    return f"{target_year - 1}-01-01", f"{target_year + 1}-01-01"

def make_ee_fc_from_points(df, id_col="traj_sample_id"):
    features = []
    for _, r in df.iterrows():
        geom = ee.Geometry.Point([float(r["longitude"]), float(r["latitude"])])
        props = {
            "traj_sample_id": str(r[id_col]),
            "unit_id": str(r["unit_id"]),
            "zone_id": str(r["zone_id"]),
            "target_year": int(r["target_year"]),
            "latitude": float(r["latitude"]),
            "longitude": float(r["longitude"]),
        }
        features.append(ee.Feature(geom, props))
    return ee.FeatureCollection(features)

def scan_npz_ids(chunk_dir, id_key="traj_sample_ids"):
    chunk_dir = Path(chunk_dir)
    ids = set()
    files = sorted(chunk_dir.glob("*.npz"))
    for fp in files:
        try:
            d = np.load(fp, allow_pickle=True)
            if id_key in d:
                ids.update([str(x) for x in d[id_key].astype(str)])
        except Exception:
            pass
    return ids, files

def summarize_array(name, arr):
    arr = np.asarray(arr)
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")
    if np.issubdtype(arr.dtype, np.number):
        print("  NaN:", int(np.isnan(arr).sum()) if np.issubdtype(arr.dtype, np.floating) else "n/a")
        print("  Inf:", int(np.isinf(arr).sum()) if np.issubdtype(arr.dtype, np.floating) else "n/a")
        try:
            print("  min/max:", float(np.nanmin(arr)), float(np.nanmax(arr)))
        except Exception:
            pass


## 4. Build trajectory master table

In [ ]:
# ============================================================
# 4. BUILD TRAJECTORY MASTER TABLE
# ============================================================

assert_exists(REFERENCE_CSV, "reference CSV")

if os.path.exists(TRAJ_MASTER_CSV) and not OVERWRITE_MASTER:
    trajectory_master_df = pd.read_csv(TRAJ_MASTER_CSV)
    print("Reloaded existing trajectory master:")
    print(TRAJ_MASTER_CSV)

else:
    ref_df = pd.read_csv(REFERENCE_CSV)

    print("Reference rows:", len(ref_df))
    print("Reference columns:", list(ref_df.columns))

    required = ["unit_id", "observation_year", "observation_month", "land_cover"]
    missing = [c for c in required if c not in ref_df.columns]
    if missing:
        raise RuntimeError("Missing required reference columns: " + ", ".join(missing))

    lon_col, lat_col = detect_lon_lat_columns(ref_df)
    zone_col = detect_zone_column(ref_df)

    if lon_col is None or lat_col is None:
        raise RuntimeError("Could not detect longitude/latitude columns.")
    if zone_col is None:
        raise RuntimeError("Could not detect zone column.")

    ref_df = ref_df.copy()
    ref_df["unit_id"] = ref_df["unit_id"].astype(str)
    ref_df["zone_id"] = ref_df[zone_col].astype(str)
    ref_df["latitude"] = pd.to_numeric(ref_df[lat_col], errors="coerce")
    ref_df["longitude"] = pd.to_numeric(ref_df[lon_col], errors="coerce")
    ref_df["observation_year"] = ref_df["observation_year"].astype(int)
    ref_df["observation_month"] = ref_df["observation_month"].astype(int)

    if "sample_id" not in ref_df.columns:
        ref_df["sample_id"] = [
            make_sample_id(u, y, m)
            for u, y, m in zip(ref_df["unit_id"], ref_df["observation_year"], ref_df["observation_month"])
        ]

    # One anchor record per unit_id.
    # If duplicates exist, keep the first but preserve anchor metadata.
    anchor_df = (
        ref_df
        .dropna(subset=["unit_id", "latitude", "longitude", "zone_id"])
        .sort_values(["unit_id", "observation_year", "observation_month"])
        .drop_duplicates("unit_id", keep="first")
        .reset_index(drop=True)
    )

    if TARGET_ZONE_FOR_TRAJ_CACHE is not None and str(TARGET_ZONE_FOR_TRAJ_CACHE).lower() not in ["all", "none", ""]:
        anchor_df = (
            anchor_df
            .loc[anchor_df["zone_id"].str.lower().eq(str(TARGET_ZONE_FOR_TRAJ_CACHE).lower())]
            .copy()
            .reset_index(drop=True)
        )

    print("Anchor points:", len(anchor_df))

    rows = []
    for _, r in anchor_df.iterrows():
        for y in TRAJ_YEARS:
            rows.append({
                "traj_sample_id": make_traj_sample_id(r["unit_id"], y),
                "unit_id": str(r["unit_id"]),
                "zone_id": str(r["zone_id"]),
                "target_year": int(y),
                "year": int(y),
                "latitude": float(r["latitude"]),
                "longitude": float(r["longitude"]),
                "anchor_land_cover": str(r["land_cover"]),
                "anchor_observation_year": int(r["observation_year"]),
                "anchor_observation_month": int(r["observation_month"]),
                "anchor_sample_id": str(r["sample_id"]),
                "gee_year": int(y),
                "gee_patch_year": int(y),
                "s2_temporal_mode": S2_TEMPORAL_MODE,
                "s2_start_date": get_s2_window_for_target_year(y)[0],
                "s2_end_date_exclusive": get_s2_window_for_target_year(y)[1],
                "s2_n_months_expected": S2_TEMPORAL_LENGTH,
                "s2_month_starts": "|".join(get_s2_month_starts_for_target_year(y)),
                "unit_year_key": f"{str(r['unit_id'])}_{int(y)}",
            })

    trajectory_master_df = pd.DataFrame(rows)

    # Add split info if master split exists, using anchor sample_id where relevant.
    if os.path.exists(MASTER_SPLIT_PATH):
        split_df = pd.read_csv(MASTER_SPLIT_PATH)
        if "sample_id" in split_df.columns and "split" in split_df.columns:
            split_small = split_df[["sample_id", "split"]].copy()
            split_small["sample_id"] = split_small["sample_id"].astype(str)
            trajectory_master_df = trajectory_master_df.merge(
                split_small.rename(columns={"sample_id": "anchor_sample_id", "split": "anchor_split"}),
                on="anchor_sample_id",
                how="left",
            )

    trajectory_master_df.to_csv(TRAJ_MASTER_CSV, index=False)
    print("Saved trajectory master:")
    print(TRAJ_MASTER_CSV)

trajectory_master_df["traj_sample_id"] = trajectory_master_df["traj_sample_id"].astype(str)
trajectory_master_df["unit_id"] = trajectory_master_df["unit_id"].astype(str)
trajectory_master_df["zone_id"] = trajectory_master_df["zone_id"].astype(str)
trajectory_master_df["target_year"] = trajectory_master_df["target_year"].astype(int)

print("\ntrajectory_master_df:", trajectory_master_df.shape)
display(trajectory_master_df.head())
display(trajectory_master_df.groupby("target_year").size().reset_index(name="n_rows"))
display(trajectory_master_df.groupby("zone_id").size().reset_index(name="n_rows"))


OK - reference CSV:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Kasai_reference.csv
Reloaded existing trajectory master:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/trajectory_master_all_zones_2017_2025.csv

trajectory_master_df: (33354, 20)


,traj_sample_id,unit_id,zone_id,target_year,year,latitude,longitude,anchor_land_cover,anchor_observation_year,anchor_observation_month,anchor_sample_id,gee_year,gee_patch_year,s2_temporal_mode,s2_start_date,s2_end_date_exclusive,s2_n_months_expected,s2_month_starts,unit_year_key,anchor_split
0,KASAI_000001_2017,KASAI_000001,Ilebo,2017,2017,-4.336549,20.591859,built_up,2020,4,KASAI_000001_2020_04,2017,2017,prev_plus_target_year,2016-01-01,2018-01-01,24,2016-01-01|2016-02-01|2016-03-01|2016-04-01|20...,KASAI_000001_2017,train
1,KASAI_000001_2018,KASAI_000001,Ilebo,2018,2018,-4.336549,20.591859,built_up,2020,4,KASAI_000001_2020_04,2018,2018,prev_plus_target_year,2017-01-01,2019-01-01,24,2017-01-01|2017-02-01|2017-03-01|2017-04-01|20...,KASAI_000001_2018,train
2,KASAI_000001_2019,KASAI_000001,Ilebo,2019,2019,-4.336549,20.591859,built_up,2020,4,KASAI_000001_2020_04,2019,2019,prev_plus_target_year,2018-01-01,2020-01-01,24,2018-01-01|2018-02-01|2018-03-01|2018-04-01|20...,KASAI_000001_2019,train
3,KASAI_000001_2020,KASAI_000001,Ilebo,2020,2020,-4.336549,20.591859,built_up,2020,4,KASAI_000001_2020_04,2020,2020,prev_plus_target_year,2019-01-01,2021-01-01,24,2019-01-01|2019-02-01|2019-03-01|2019-04-01|20...,KASAI_000001_2020,train
4,KASAI_000001_2021,KASAI_000001,Ilebo,2021,2021,-4.336549,20.591859,built_up,2020,4,KASAI_000001_2020_04,2021,2021,prev_plus_target_year,2020-01-01,2022-01-01,24,2020-01-01|2020-02-01|2020-03-01|2020-04-01|20...,KASAI_000001_2021,train


,target_year,n_rows
0,2017,3706
1,2018,3706
2,2019,3706
3,2020,3706
4,2021,3706
5,2022,3706
6,2023,3706
7,2024,3706
8,2025,3706


,zone_id,n_rows
0,Ilebo,11862
1,Mbuji,7551
2,Mushie,11466
3,Tshikapa,2475


## 5. Extract GEE annual point embeddings (`n × 64`)

In [ ]:
# ============================================================
# 5. EXTRACT GEE ANNUAL POINT EMBEDDINGS
# ============================================================

print("Launching GEE annual point embedding extraction")
print("Output:", TRAJ_GEE_POINT_NPZ)

if os.path.exists(TRAJ_GEE_POINT_NPZ) and not OVERWRITE_GEE_POINT_FINAL:
    print("\nFinal GEE point cache already exists. Reloading:")
    print(TRAJ_GEE_POINT_NPZ)
    d = np.load(TRAJ_GEE_POINT_NPZ, allow_pickle=True)
    X_traj_gee_point = d["X_gee_point"].astype(np.float32)
    ids_traj_gee_point = d["traj_sample_ids"].astype(str)

else:
    gee_point_chunk_dir = os.path.join(GEE_POINT_DIR, "chunks")
    Path(gee_point_chunk_dir).mkdir(parents=True, exist_ok=True)

    existing_ids, existing_files = scan_npz_ids(gee_point_chunk_dir, "traj_sample_ids")
    print("Existing GEE point chunks:", len(existing_files))
    print("Existing extracted IDs:", len(existing_ids))

    annual_ic = ee.ImageCollection(GEE_EMBEDDING_COLLECTION)

    def get_annual_embedding_image(year):
        start = ee.Date.fromYMD(int(year), 1, 1)
        end = start.advance(1, "year")
        return annual_ic.filterDate(start, end).mosaic().toFloat()

    embedding_bands = get_annual_embedding_image(TRAJ_START_YEAR).bandNames().getInfo()
    print("Embedding bands:", len(embedding_bands), embedding_bands[:5], "...")

    for year in sorted(trajectory_master_df["target_year"].unique()):
        year = int(year)
        year_df = (
            trajectory_master_df
            .loc[trajectory_master_df["target_year"].eq(year)]
            .sort_values("traj_sample_id")
            .reset_index(drop=True)
            .copy()
        )

        n_year = len(year_df)
        n_chunks = math.ceil(n_year / EE_BATCH_SIZE_GEE_POINT)

        print("\n" + "=" * 80)
        print(f"Extracting GEE point embeddings for target year {year}")
        print("Rows:", n_year, "| chunks:", n_chunks)
        print("=" * 80)

        for chunk_i in range(n_chunks):
            start_i = chunk_i * EE_BATCH_SIZE_GEE_POINT
            end_i = min((chunk_i + 1) * EE_BATCH_SIZE_GEE_POINT, n_year)

            df_chunk = year_df.iloc[start_i:end_i].copy()
            chunk_ids = set(df_chunk["traj_sample_id"].astype(str))

            if (not OVERWRITE_GEE_POINT_CHUNKS) and chunk_ids.issubset(existing_ids):
                print(f"  skip rows {start_i}:{end_i} all extracted")
                continue

            chunk_path = os.path.join(
                gee_point_chunk_dir,
                f"gee_point_year{year}_rows{start_i}_{end_i}_batch{EE_BATCH_SIZE_GEE_POINT}.npz"
            )

            print(f"  extracting rows {start_i}:{end_i} | n={len(df_chunk)}")

            try:
                fc = make_ee_fc_from_points(df_chunk)
                img = get_annual_embedding_image(year).select(embedding_bands)

                sampled = img.sampleRegions(
                    collection=fc,
                    properties=["traj_sample_id", "unit_id", "zone_id", "target_year", "latitude", "longitude"],
                    scale=GEE_SCALE,
                    geometries=False,
                    tileScale=EE_TILE_SCALE_GEE,
                )

                info = sampled.getInfo()

                rows = []
                X = []

                for f in info.get("features", []):
                    props = f.get("properties", {})
                    vals = []
                    ok = True
                    for b in embedding_bands:
                        v = props.get(b, None)
                        if v is None:
                            ok = False
                            break
                        vals.append(float(v))
                    if not ok:
                        continue

                    X.append(vals)
                    rows.append({
                        "traj_sample_id": str(props["traj_sample_id"]),
                        "unit_id": str(props["unit_id"]),
                        "zone_id": str(props["zone_id"]),
                        "target_year": int(props["target_year"]),
                        "latitude": float(props["latitude"]),
                        "longitude": float(props["longitude"]),
                    })

                if len(X) == 0:
                    print("    WARNING: no rows returned")
                    continue

                meta = pd.DataFrame(rows)
                X = np.asarray(X, dtype=np.float32)

                np.savez_compressed(
                    chunk_path,
                    X=X,
                    traj_sample_ids=meta["traj_sample_id"].astype(str).values,
                    unit_ids=meta["unit_id"].astype(str).values,
                    zone_id=meta["zone_id"].astype(str).values,
                    target_year=meta["target_year"].astype(np.int16).values,
                    latitude=meta["latitude"].astype(np.float64).values,
                    longitude=meta["longitude"].astype(np.float64).values,
                    embedding_bands=np.asarray(embedding_bands).astype(str),
                    source=GEE_EMBEDDING_COLLECTION,
                    target_year_alignment="annual embedding image at target_year",
                )

                existing_ids.update(meta["traj_sample_id"].astype(str).tolist())
                print("    saved:", os.path.basename(chunk_path), X.shape)

            except Exception as e:
                print("    WARNING: extraction failed:", repr(e))
                print("    Chunk left unsaved so it can be retried.")

            time.sleep(EE_SLEEP_SECONDS)

    # Assemble final in master order
    print("\nAssembling final GEE point cache...")

    chunk_files = sorted(Path(gee_point_chunk_dir).glob("*.npz"))
    if len(chunk_files) == 0:
        raise RuntimeError("No GEE point chunks found.")

    X_parts = []
    meta_parts = []
    offset = 0

    for fp in tqdm(chunk_files, desc="Assembling GEE point chunks"):
        d = np.load(fp, allow_pickle=True)
        X = d["X"].astype(np.float32)
        meta = pd.DataFrame({
            "traj_sample_id": d["traj_sample_ids"].astype(str),
            "unit_id": d["unit_ids"].astype(str),
            "zone_id": d["zone_id"].astype(str),
            "target_year": d["target_year"].astype(int),
            "latitude": d["latitude"].astype(float),
            "longitude": d["longitude"].astype(float),
            "extract_idx": np.arange(offset, offset + len(X)),
        })
        X_parts.append(X)
        meta_parts.append(meta)
        offset += len(X)

    X_all = np.concatenate(X_parts, axis=0).astype(np.float32)
    meta_all = pd.concat(meta_parts, ignore_index=True)
    meta_all["extract_idx"] = np.arange(len(meta_all))

    # De-duplicate IDs, keep last extracted version
    meta_all["_row"] = np.arange(len(meta_all))
    meta_all = meta_all.sort_values("_row").drop_duplicates("traj_sample_id", keep="last").drop(columns="_row")

    order_df = trajectory_master_df[["traj_sample_id", "unit_id", "zone_id", "target_year", "latitude", "longitude"]].copy()
    order_df["order_idx"] = np.arange(len(order_df))

    joined = (
        order_df
        .merge(meta_all[["traj_sample_id", "extract_idx"]], on="traj_sample_id", how="inner")
        .sort_values("order_idx")
        .reset_index(drop=True)
    )

    missing_ids = sorted(set(order_df["traj_sample_id"]) - set(joined["traj_sample_id"]))
    print("Requested rows:", len(order_df))
    print("Extracted rows:", len(joined))
    print("Missing rows:", len(missing_ids))
    if missing_ids:
        print("First missing:", missing_ids[:20])

    idx = joined["extract_idx"].to_numpy().astype(int)
    X_traj_gee_point = X_all[idx].astype(np.float32)
    ids_traj_gee_point = joined["traj_sample_id"].astype(str).values

    np.savez_compressed(
        TRAJ_GEE_POINT_NPZ,
        X_gee_point=X_traj_gee_point,
        traj_sample_ids=ids_traj_gee_point,
        unit_ids=joined["unit_id"].astype(str).values,
        zone_id=joined["zone_id"].astype(str).values,
        target_year=joined["target_year"].astype(np.int16).values,
        latitude=joined["latitude"].astype(np.float64).values,
        longitude=joined["longitude"].astype(np.float64).values,
        embedding_bands=np.asarray(embedding_bands).astype(str),
        source=GEE_EMBEDDING_COLLECTION,
        target_year_alignment="annual embedding image at target_year",
    )

    print("Saved final GEE point cache:")
    print(TRAJ_GEE_POINT_NPZ)

summarize_array("X_traj_gee_point", X_traj_gee_point)
print("ids_traj_gee_point:", ids_traj_gee_point.shape)


Launching GEE annual point embedding extraction
Output: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/gee_embedding_point/trajectory_gee_embedding_point_all_zones_2017_2025.npz
Existing GEE point chunks: 0
Existing extracted IDs: 0
Embedding bands: 64 ['A00', 'A01', 'A02', 'A03', 'A04'] ...

Extracting GEE point embeddings for target year 2017
Rows: 3706 | chunks: 19
  extracting rows 0:200 | n=200
    saved: gee_point_year2017_rows0_200_batch200.npz (200, 64)
  extracting rows 200:400 | n=200
    saved: gee_point_year2017_rows200_400_batch200.npz (200, 64)
  extracting rows 400:600 | n=200
    saved: gee_point_year2017_rows400_600_batch200.npz (200, 64)
  extracting rows 600:800 | n=200
    saved: gee_point_year2017_rows600_800_batch200.npz (200, 64)
  extracting rows 800:1000 | n=200
    saved: gee_point_year2017_rows800_1000_batch200.npz (200, 64)
  extracting rows 1000:1200 | n=200
    saved: gee_point_year2017_rows1000_1200_batch200.npz

Assembling GEE point chunks:   0%|          | 0/171 [00:00<?, ?it/s]

Requested rows: 33354
Extracted rows: 33354
Missing rows: 0
Saved final GEE point cache:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/gee_embedding_point/trajectory_gee_embedding_point_all_zones_2017_2025.npz
X_traj_gee_point: shape=(33354, 64), dtype=float32
  NaN: 0
  Inf: 0
  min/max: -0.45496347546577454 0.45496347546577454
ids_traj_gee_point: (33354,)


## 6. Extract GEE annual p3 embedding patches for CNN-GEE

In [ ]:
# ============================================================
# 6. EXTRACT GEE ANNUAL P3 EMBEDDING PATCHES
# ============================================================

print("Launching GEE annual p3 patch extraction")
print("Output:", TRAJ_GEE_PATCH_NPZ)

if os.path.exists(TRAJ_GEE_PATCH_NPZ) and not OVERWRITE_GEE_PATCH_FINAL:
    print("\nFinal GEE patch cache already exists. Reloading:")
    print(TRAJ_GEE_PATCH_NPZ)
    d = np.load(TRAJ_GEE_PATCH_NPZ, allow_pickle=True)
    X_traj_gee_patch = d["X_gee_patch"].astype(np.float32)
    ids_traj_gee_patch = d["traj_sample_ids"].astype(str)

else:
    gee_patch_chunk_dir = os.path.join(GEE_PATCH_DIR, "chunks")
    Path(gee_patch_chunk_dir).mkdir(parents=True, exist_ok=True)

    existing_ids, existing_files = scan_npz_ids(gee_patch_chunk_dir, "traj_sample_ids")
    print("Existing GEE patch chunks:", len(existing_files))
    print("Existing extracted IDs:", len(existing_ids))

    annual_ic = ee.ImageCollection(GEE_EMBEDDING_COLLECTION)

    def get_annual_embedding_image(year):
        start = ee.Date.fromYMD(int(year), 1, 1)
        end = start.advance(1, "year")
        return annual_ic.filterDate(start, end).mosaic().toFloat()

    embedding_bands = get_annual_embedding_image(TRAJ_START_YEAR).bandNames().getInfo()
    print("Embedding bands:", len(embedding_bands), embedding_bands[:5], "...")

    # p3 centered kernel.
    # This is the same concept as training: 3 x 3 neighborhood around the point pixel.
    kernel_weights = [[1 for _ in range(GEE_PATCH_SIZE)] for _ in range(GEE_PATCH_SIZE)]
    kernel = ee.Kernel.fixed(
        width=GEE_PATCH_SIZE,
        height=GEE_PATCH_SIZE,
        weights=kernel_weights,
        x=-(GEE_PATCH_SIZE // 2),
        y=-(GEE_PATCH_SIZE // 2),
        normalize=False,
    )

    def extract_patch_from_feature_properties(props, band_names, patch_size):
        band_arrays = []
        for b in band_names:
            if b not in props:
                return None
            arr = np.asarray(props[b], dtype=np.float32)
            if arr.shape != (patch_size, patch_size):
                return None
            band_arrays.append(arr)
        return np.stack(band_arrays, axis=-1).astype(np.float32)

    for year in sorted(trajectory_master_df["target_year"].unique()):
        year = int(year)
        year_df = (
            trajectory_master_df
            .loc[trajectory_master_df["target_year"].eq(year)]
            .sort_values("traj_sample_id")
            .reset_index(drop=True)
            .copy()
        )

        n_year = len(year_df)
        n_chunks = math.ceil(n_year / EE_BATCH_SIZE_GEE_PATCH)

        print("\n" + "=" * 80)
        print(f"Extracting GEE p{GEE_PATCH_SIZE} patches for target year {year}")
        print("Rows:", n_year, "| chunks:", n_chunks)
        print("=" * 80)

        for chunk_i in range(n_chunks):
            start_i = chunk_i * EE_BATCH_SIZE_GEE_PATCH
            end_i = min((chunk_i + 1) * EE_BATCH_SIZE_GEE_PATCH, n_year)

            df_chunk = year_df.iloc[start_i:end_i].copy()
            chunk_ids = set(df_chunk["traj_sample_id"].astype(str))

            if (not OVERWRITE_GEE_PATCH_CHUNKS) and chunk_ids.issubset(existing_ids):
                print(f"  skip rows {start_i}:{end_i} all extracted")
                continue

            chunk_path = os.path.join(
                gee_patch_chunk_dir,
                f"gee_patch_p{GEE_PATCH_SIZE}_year{year}_rows{start_i}_{end_i}_batch{EE_BATCH_SIZE_GEE_PATCH}.npz"
            )

            print(f"  extracting rows {start_i}:{end_i} | n={len(df_chunk)}")

            try:
                fc = make_ee_fc_from_points(df_chunk)
                img = get_annual_embedding_image(year).select(embedding_bands)
                arr_img = img.neighborhoodToArray(kernel)

                sampled = arr_img.sampleRegions(
                    collection=fc,
                    properties=["traj_sample_id", "unit_id", "zone_id", "target_year", "latitude", "longitude"],
                    scale=GEE_SCALE,
                    geometries=False,
                    tileScale=EE_TILE_SCALE_GEE,
                )

                info = sampled.getInfo()

                rows = []
                X = []

                for f in info.get("features", []):
                    props = f.get("properties", {})
                    patch = extract_patch_from_feature_properties(props, embedding_bands, GEE_PATCH_SIZE)
                    if patch is None:
                        continue
                    if not np.isfinite(patch).any():
                        continue

                    X.append(patch)
                    rows.append({
                        "traj_sample_id": str(props["traj_sample_id"]),
                        "unit_id": str(props["unit_id"]),
                        "zone_id": str(props["zone_id"]),
                        "target_year": int(props["target_year"]),
                        "latitude": float(props["latitude"]),
                        "longitude": float(props["longitude"]),
                    })

                if len(X) == 0:
                    print("    WARNING: no rows returned")
                    continue

                meta = pd.DataFrame(rows)
                X = np.stack(X, axis=0).astype(np.float32)

                np.savez_compressed(
                    chunk_path,
                    X=X,
                    traj_sample_ids=meta["traj_sample_id"].astype(str).values,
                    unit_ids=meta["unit_id"].astype(str).values,
                    zone_id=meta["zone_id"].astype(str).values,
                    target_year=meta["target_year"].astype(np.int16).values,
                    latitude=meta["latitude"].astype(np.float64).values,
                    longitude=meta["longitude"].astype(np.float64).values,
                    embedding_bands=np.asarray(embedding_bands).astype(str),
                    patch_size=np.asarray([GEE_PATCH_SIZE], dtype=np.int16),
                    source=GEE_EMBEDDING_COLLECTION,
                    target_year_alignment="annual embedding patch at target_year",
                )

                existing_ids.update(meta["traj_sample_id"].astype(str).tolist())
                print("    saved:", os.path.basename(chunk_path), X.shape)

            except Exception as e:
                print("    WARNING: extraction failed:", repr(e))
                print("    Chunk left unsaved so it can be retried.")

            time.sleep(EE_SLEEP_SECONDS)

    # Assemble final in master order
    print("\nAssembling final GEE patch cache...")

    chunk_files = sorted(Path(gee_patch_chunk_dir).glob("*.npz"))
    if len(chunk_files) == 0:
        raise RuntimeError("No GEE patch chunks found.")

    X_parts = []
    meta_parts = []
    offset = 0

    for fp in tqdm(chunk_files, desc="Assembling GEE patch chunks"):
        d = np.load(fp, allow_pickle=True)
        X = d["X"].astype(np.float32)
        meta = pd.DataFrame({
            "traj_sample_id": d["traj_sample_ids"].astype(str),
            "unit_id": d["unit_ids"].astype(str),
            "zone_id": d["zone_id"].astype(str),
            "target_year": d["target_year"].astype(int),
            "latitude": d["latitude"].astype(float),
            "longitude": d["longitude"].astype(float),
            "extract_idx": np.arange(offset, offset + len(X)),
        })
        X_parts.append(X)
        meta_parts.append(meta)
        offset += len(X)

    X_all = np.concatenate(X_parts, axis=0).astype(np.float32)
    meta_all = pd.concat(meta_parts, ignore_index=True)
    meta_all["extract_idx"] = np.arange(len(meta_all))

    meta_all["_row"] = np.arange(len(meta_all))
    meta_all = meta_all.sort_values("_row").drop_duplicates("traj_sample_id", keep="last").drop(columns="_row")

    order_df = trajectory_master_df[["traj_sample_id", "unit_id", "zone_id", "target_year", "latitude", "longitude"]].copy()
    order_df["order_idx"] = np.arange(len(order_df))

    joined = (
        order_df
        .merge(meta_all[["traj_sample_id", "extract_idx"]], on="traj_sample_id", how="inner")
        .sort_values("order_idx")
        .reset_index(drop=True)
    )

    missing_ids = sorted(set(order_df["traj_sample_id"]) - set(joined["traj_sample_id"]))
    print("Requested rows:", len(order_df))
    print("Extracted rows:", len(joined))
    print("Missing rows:", len(missing_ids))
    if missing_ids:
        print("First missing:", missing_ids[:20])

    idx = joined["extract_idx"].to_numpy().astype(int)
    X_traj_gee_patch = X_all[idx].astype(np.float32)
    ids_traj_gee_patch = joined["traj_sample_id"].astype(str).values

    np.savez_compressed(
        TRAJ_GEE_PATCH_NPZ,
        X_gee_patch=X_traj_gee_patch,
        traj_sample_ids=ids_traj_gee_patch,
        unit_ids=joined["unit_id"].astype(str).values,
        zone_id=joined["zone_id"].astype(str).values,
        target_year=joined["target_year"].astype(np.int16).values,
        latitude=joined["latitude"].astype(np.float64).values,
        longitude=joined["longitude"].astype(np.float64).values,
        embedding_bands=np.asarray(embedding_bands).astype(str),
        patch_size=np.asarray([GEE_PATCH_SIZE], dtype=np.int16),
        source=GEE_EMBEDDING_COLLECTION,
        target_year_alignment="annual embedding patch at target_year",
    )

    print("Saved final GEE patch cache:")
    print(TRAJ_GEE_PATCH_NPZ)

summarize_array("X_traj_gee_patch", X_traj_gee_patch)
print("ids_traj_gee_patch:", ids_traj_gee_patch.shape)


Launching GEE annual p3 patch extraction
Output: /content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/gee_embedding_patch_p3/trajectory_gee_embedding_patch_p3_all_zones_2017_2025.npz
Existing GEE patch chunks: 0
Existing extracted IDs: 0
Embedding bands: 64 ['A00', 'A01', 'A02', 'A03', 'A04'] ...

Extracting GEE p3 patches for target year 2017
Rows: 3706 | chunks: 38
  extracting rows 0:100 | n=100
    saved: gee_patch_p3_year2017_rows0_100_batch100.npz (100, 3, 3, 64)
  extracting rows 100:200 | n=100
    saved: gee_patch_p3_year2017_rows100_200_batch100.npz (100, 3, 3, 64)
  extracting rows 200:300 | n=100
    saved: gee_patch_p3_year2017_rows200_300_batch100.npz (100, 3, 3, 64)
  extracting rows 300:400 | n=100
    saved: gee_patch_p3_year2017_rows300_400_batch100.npz (100, 3, 3, 64)
  extracting rows 400:500 | n=100
    saved: gee_patch_p3_year2017_rows400_500_batch100.npz (100, 3, 3, 64)
  extracting rows 500:600 | n=100
    saved: gee_patch_p

Assembling GEE patch chunks:   0%|          | 0/342 [00:00<?, ?it/s]

Requested rows: 33354
Extracted rows: 33354
Missing rows: 0
Saved final GEE patch cache:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/gee_embedding_patch_p3/trajectory_gee_embedding_patch_p3_all_zones_2017_2025.npz
X_traj_gee_patch: shape=(33354, 3, 3, 64), dtype=float32
  NaN: 0
  Inf: 0
  min/max: -0.476370632648468 0.4656055271625519
ids_traj_gee_patch: (33354,)


## 7. Extract Sentinel-2 temporal T24 (`n × 24 × 11`)

In [ ]:
# ============================================================
# 7. EXTRACT SENTINEL-2 TEMPORAL T24
# ============================================================

print("Launching Sentinel-2 temporal T24 extraction")
print("Output:", TRAJ_S2_TEMPORAL_NPZ)
print("S2 temporal alignment: Jan Y-1 to Dec Y")
print("S2 features:", S2_FEATURE_BANDS)

if os.path.exists(TRAJ_S2_TEMPORAL_NPZ) and not OVERWRITE_S2_FINAL:
    print("\nFinal S2 temporal cache already exists. Reloading:")
    print(TRAJ_S2_TEMPORAL_NPZ)
    d = np.load(TRAJ_S2_TEMPORAL_NPZ, allow_pickle=True)
    X_traj_s2_temporal = d["X_s2_temporal"].astype(np.float32)
    ids_traj_s2_temporal = d["traj_sample_ids"].astype(str)

else:
    s2_chunk_dir = os.path.join(S2_TEMPORAL_DIR, "chunks")
    Path(s2_chunk_dir).mkdir(parents=True, exist_ok=True)

    existing_ids, existing_files = scan_npz_ids(s2_chunk_dir, "traj_sample_ids")
    print("Existing S2 chunks:", len(existing_files))
    print("Existing extracted IDs:", len(existing_ids))

    def add_s2_indices(img):
        # Assumes selected optical bands are already scaled to reflectance.
        b2 = img.select("B2")
        b3 = img.select("B3")
        b4 = img.select("B4")
        b8 = img.select("B8")
        b11 = img.select("B11")
        b12 = img.select("B12")
        eps = ee.Image.constant(1e-6)

        ndvi = b8.subtract(b4).divide(b8.add(b4).add(eps)).rename("NDVI")
        evi = (
            b8.subtract(b4)
            .multiply(2.5)
            .divide(
                b8
                .add(b4.multiply(6.0))
                .subtract(b2.multiply(7.5))
                .add(1.0)
                .add(eps)
            )
            .rename("EVI")
        )
        ndwi = b3.subtract(b8).divide(b3.add(b8).add(eps)).rename("NDWI")
        nbr = b8.subtract(b12).divide(b8.add(b12).add(eps)).rename("NBR")

        return img.addBands([ndvi, evi, ndwi, nbr])

    def mask_s2_sr_with_scl(img):
        # Notebook-2 style: exclude cloud shadow/cloud/cirrus/snow.
        # Do not exclude 0/1 here, to keep logic as close as possible to original cache.
        scl = img.select("SCL")
        valid = (
            scl.neq(3)      # cloud shadow
            .And(scl.neq(8))    # cloud medium probability
            .And(scl.neq(9))    # cloud high probability
            .And(scl.neq(10))   # thin cirrus
            .And(scl.neq(11))   # snow/ice
        )
        return img.updateMask(valid)

    def join_s2_cloud_probability_inner(s2_sr_col, s2_cloud_col):
        # Notebook-2 style inner join by system:index.
        joined = ee.Join.inner().apply(
            primary=s2_sr_col,
            secondary=s2_cloud_col,
            condition=ee.Filter.equals(
                leftField="system:index",
                rightField="system:index",
            ),
        )

        def merge_joined(f):
            primary = ee.Image(f.get("primary"))
            secondary = ee.Image(f.get("secondary"))
            return primary.set("cloud_probability_image", secondary)

        return ee.ImageCollection(joined.map(merge_joined))

    def mask_s2_cloud_probability(img):
        cloud_img = ee.Image(img.get("cloud_probability_image"))
        cloud_prob = cloud_img.select("probability")
        clear = cloud_prob.lt(S2_CLOUD_PROB_THRESHOLD)
        return img.updateMask(clear)

    def prepare_s2_image(img):
        img = mask_s2_sr_with_scl(img)
        img = mask_s2_cloud_probability(img)

        optical = (
            img
            .select(S2_RAW_BANDS)
            .multiply(0.0001)
            .toFloat()
        )

        optical = add_s2_indices(optical)
        return optical.select(S2_RAW_BANDS + S2_INDEX_BANDS).toFloat()

    def monthly_s2_composite(year, month, region):
        year = int(year)
        month = int(month)

        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, "month")

        s2_sr = (
            ee.ImageCollection(S2_SR_IC)
            .filterDate(start, end)
            .filterBounds(region)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_CLOUDY_PIXEL_PERCENTAGE_MAX))
        )

        s2_cloud = (
            ee.ImageCollection(S2_CLOUD_PROB_IC)
            .filterDate(start, end)
            .filterBounds(region)
        )

        joined = join_s2_cloud_probability_inner(s2_sr, s2_cloud)
        prepared = joined.map(prepare_s2_image)

        n = prepared.size()

        spectral_empty = (
            ee.Image.constant([S2_NODATA_VALUE] * len(S2_RAW_BANDS + S2_INDEX_BANDS))
            .rename(S2_RAW_BANDS + S2_INDEX_BANDS)
            .toFloat()
        )

        composite = ee.Image(
            ee.Algorithms.If(
                n.gt(0),
                prepared.median().toFloat(),
                spectral_empty,
            )
        )

        # Notebook-2 style: valid_obs_count from NDVI count.
        valid_count = ee.Image(
            ee.Algorithms.If(
                n.gt(0),
                prepared.select("NDVI").count().rename("valid_obs_count").toFloat(),
                ee.Image.constant(0).rename("valid_obs_count").toFloat(),
            )
        )

        # Important for point-year completeness: unmask only after count is created.
        composite = composite.unmask(S2_NODATA_VALUE).toFloat()
        valid_count = valid_count.unmask(0).toFloat()

        return composite.addBands(valid_count).select(S2_FEATURE_BANDS).toFloat()

    def get_s2_temporal_image_for_target_year(target_year, region):
        month_starts = get_s2_month_starts_for_target_year(target_year)

        monthly_images = []
        for t, date_str in enumerate(month_starts):
            yy = int(date_str[:4])
            mm = int(date_str[5:7])

            img_month = monthly_s2_composite(yy, mm, region)
            img_month = img_month.rename([f"t{t:02d}_{b}" for b in S2_FEATURE_BANDS])
            monthly_images.append(img_month)

        return ee.Image.cat(monthly_images).toFloat()

    def extract_s2_temporal_chunk(df_chunk, target_year):
        fc = make_ee_fc_from_points(df_chunk)
        region = fc.geometry().bounds().buffer(1000)

        img = get_s2_temporal_image_for_target_year(target_year, region)

        sampled = img.sampleRegions(
            collection=fc,
            properties=["traj_sample_id", "unit_id", "zone_id", "target_year", "latitude", "longitude"],
            scale=S2_SCALE,
            geometries=False,
            tileScale=EE_TILE_SCALE_S2,
        )

        info = sampled.getInfo()

        X_list = []
        rows = []
        valid_months_list = []
        missing_fraction_list = []
        valid_obs_sum_list = []

        for f in info.get("features", []):
            props = f.get("properties", {})

            arr = np.full(
                (S2_TEMPORAL_LENGTH, len(S2_FEATURE_BANDS)),
                np.nan,
                dtype=np.float32,
            )

            for t in range(S2_TEMPORAL_LENGTH):
                for j, b in enumerate(S2_FEATURE_BANDS):
                    key = f"t{t:02d}_{b}"
                    if key in props and props[key] is not None:
                        arr[t, j] = np.float32(props[key])

            # Convert explicit nodata back to NaN for spectral/index bands.
            spectral_part = arr[:, :-1]
            spectral_part[spectral_part <= S2_NODATA_VALUE / 2] = np.nan
            arr[:, :-1] = spectral_part

            # valid_obs_count remains numeric; replace NaN by 0 if needed
            valid_counts = arr[:, -1]
            valid_counts = np.where(np.isfinite(valid_counts), valid_counts, 0).astype(np.float32)
            arr[:, -1] = valid_counts

            valid_months = int(np.sum(valid_counts > 0))
            missing_fraction = float(1.0 - valid_months / S2_TEMPORAL_LENGTH)
            valid_obs_sum = float(np.sum(valid_counts))

            X_list.append(arr)

            rows.append({
                "traj_sample_id": str(props["traj_sample_id"]),
                "unit_id": str(props["unit_id"]),
                "zone_id": str(props["zone_id"]),
                "target_year": int(props["target_year"]),
                "latitude": float(props["latitude"]),
                "longitude": float(props["longitude"]),
            })

            valid_months_list.append(valid_months)
            missing_fraction_list.append(missing_fraction)
            valid_obs_sum_list.append(valid_obs_sum)

        if len(X_list) == 0:
            return None

        meta = pd.DataFrame(rows)
        meta["s2_valid_months"] = valid_months_list
        meta["s2_missing_fraction"] = missing_fraction_list
        meta["s2_valid_obs_count_sum"] = valid_obs_sum_list

        return {
            "X": np.stack(X_list, axis=0).astype(np.float32),
            "meta": meta,
        }

    for year in sorted(trajectory_master_df["target_year"].unique()):
        year = int(year)

        year_df = (
            trajectory_master_df
            .loc[trajectory_master_df["target_year"].eq(year)]
            .sort_values("traj_sample_id")
            .reset_index(drop=True)
            .copy()
        )

        n_year = len(year_df)
        n_chunks = math.ceil(n_year / EE_BATCH_SIZE_S2)
        s2_window_start, s2_window_end = get_s2_window_for_target_year(year)

        print("\n" + "=" * 80)
        print(f"Extracting S2 temporal T24 for target year {year}")
        print(f"S2 window: {s2_window_start} to {s2_window_end} exclusive")
        print("Rows:", n_year, "| chunks:", n_chunks)
        print("=" * 80)

        for chunk_i in range(n_chunks):
            start_i = chunk_i * EE_BATCH_SIZE_S2
            end_i = min((chunk_i + 1) * EE_BATCH_SIZE_S2, n_year)

            df_chunk = year_df.iloc[start_i:end_i].copy()
            chunk_ids = set(df_chunk["traj_sample_id"].astype(str))

            if (not OVERWRITE_S2_CHUNKS) and chunk_ids.issubset(existing_ids):
                print(f"  skip rows {start_i}:{end_i} all extracted")
                continue

            chunk_path = os.path.join(
                s2_chunk_dir,
                f"s2_T{S2_TEMPORAL_LENGTH}_year{year}_rows{start_i}_{end_i}_batch{EE_BATCH_SIZE_S2}.npz"
            )

            print(f"  extracting rows {start_i}:{end_i} | n={len(df_chunk)}")

            try:
                result = extract_s2_temporal_chunk(df_chunk, year)

                if result is None:
                    print("    WARNING: no rows returned")
                    continue

                meta = result["meta"]
                X = result["X"]

                np.savez_compressed(
                    chunk_path,
                    X=X,
                    traj_sample_ids=meta["traj_sample_id"].astype(str).values,
                    unit_ids=meta["unit_id"].astype(str).values,
                    zone_id=meta["zone_id"].astype(str).values,
                    target_year=meta["target_year"].astype(np.int16).values,
                    latitude=meta["latitude"].astype(np.float64).values,
                    longitude=meta["longitude"].astype(np.float64).values,
                    s2_valid_months=meta["s2_valid_months"].astype(np.int16).values,
                    s2_missing_fraction=meta["s2_missing_fraction"].astype(np.float32).values,
                    s2_valid_obs_count_sum=meta["s2_valid_obs_count_sum"].astype(np.float32).values,
                    feature_names=np.asarray(S2_FEATURE_BANDS).astype(str),
                    temporal_mode=str(S2_TEMPORAL_MODE),
                    cloud_prob_threshold=np.asarray([S2_CLOUD_PROB_THRESHOLD]),
                    cloudy_pixel_percentage_max=np.asarray([S2_CLOUDY_PIXEL_PERCENTAGE_MAX]),
                    nodata_value=np.asarray([S2_NODATA_VALUE], dtype=np.float32),
                    target_year_alignment="S2 temporal T24 from Jan Y-1 to Dec Y",
                )

                existing_ids.update(meta["traj_sample_id"].astype(str).tolist())
                print("    saved:", os.path.basename(chunk_path), X.shape)
                print(
                    "    valid months median/min/max:",
                    float(np.median(meta["s2_valid_months"])),
                    int(np.min(meta["s2_valid_months"])),
                    int(np.max(meta["s2_valid_months"])),
                )

            except Exception as e:
                print("    WARNING: extraction failed:", repr(e))
                print("    Chunk left unsaved so it can be retried.")

            time.sleep(EE_SLEEP_SECONDS)

    # Assemble final in master order
    print("\nAssembling final S2 temporal cache...")

    chunk_files = sorted(Path(s2_chunk_dir).glob("*.npz"))
    if len(chunk_files) == 0:
        raise RuntimeError("No S2 temporal chunks found.")

    X_parts = []
    meta_parts = []
    offset = 0

    for fp in tqdm(chunk_files, desc="Assembling S2 chunks"):
        d = np.load(fp, allow_pickle=True)
        X = d["X"].astype(np.float32)
        meta = pd.DataFrame({
            "traj_sample_id": d["traj_sample_ids"].astype(str),
            "unit_id": d["unit_ids"].astype(str),
            "zone_id": d["zone_id"].astype(str),
            "target_year": d["target_year"].astype(int),
            "latitude": d["latitude"].astype(float),
            "longitude": d["longitude"].astype(float),
            "s2_valid_months": d["s2_valid_months"].astype(int),
            "s2_missing_fraction": d["s2_missing_fraction"].astype(float),
            "s2_valid_obs_count_sum": d["s2_valid_obs_count_sum"].astype(float),
            "extract_idx": np.arange(offset, offset + len(X)),
            "chunk_file": fp.name,
        })
        X_parts.append(X)
        meta_parts.append(meta)
        offset += len(X)

    X_all = np.concatenate(X_parts, axis=0).astype(np.float32)
    meta_all = pd.concat(meta_parts, ignore_index=True)
    meta_all["extract_idx"] = np.arange(len(meta_all))

    # De-duplicate IDs, keep last extracted version
    meta_all["_row"] = np.arange(len(meta_all))
    meta_all = meta_all.sort_values("_row").drop_duplicates("traj_sample_id", keep="last").drop(columns="_row")

    order_df = trajectory_master_df[["traj_sample_id", "unit_id", "zone_id", "target_year", "latitude", "longitude"]].copy()
    order_df["order_idx"] = np.arange(len(order_df))

    joined = (
        order_df
        .merge(
            meta_all[
                [
                    "traj_sample_id",
                    "extract_idx",
                    "s2_valid_months",
                    "s2_missing_fraction",
                    "s2_valid_obs_count_sum",
                ]
            ],
            on="traj_sample_id",
            how="inner",
        )
        .sort_values("order_idx")
        .reset_index(drop=True)
    )

    missing_ids = sorted(set(order_df["traj_sample_id"]) - set(joined["traj_sample_id"]))
    print("Requested rows:", len(order_df))
    print("Extracted rows:", len(joined))
    print("Missing rows:", len(missing_ids))
    if missing_ids:
        print("First missing:", missing_ids[:20])

    idx = joined["extract_idx"].to_numpy().astype(int)
    X_traj_s2_temporal = X_all[idx].astype(np.float32)
    ids_traj_s2_temporal = joined["traj_sample_id"].astype(str).values

    np.savez_compressed(
        TRAJ_S2_TEMPORAL_NPZ,
        X_s2_temporal=X_traj_s2_temporal,
        traj_sample_ids=ids_traj_s2_temporal,
        unit_ids=joined["unit_id"].astype(str).values,
        zone_id=joined["zone_id"].astype(str).values,
        target_year=joined["target_year"].astype(np.int16).values,
        latitude=joined["latitude"].astype(np.float64).values,
        longitude=joined["longitude"].astype(np.float64).values,
        feature_names=np.asarray(S2_FEATURE_BANDS).astype(str),
        temporal_mode=str(S2_TEMPORAL_MODE),
        temporal_length=np.asarray([S2_TEMPORAL_LENGTH]),
        s2_valid_months=joined["s2_valid_months"].astype(np.int16).values,
        s2_missing_fraction=joined["s2_missing_fraction"].astype(np.float32).values,
        s2_valid_obs_count_sum=joined["s2_valid_obs_count_sum"].astype(np.float32).values,
        cloud_prob_threshold=np.asarray([S2_CLOUD_PROB_THRESHOLD]),
        cloudy_pixel_percentage_max=np.asarray([S2_CLOUDY_PIXEL_PERCENTAGE_MAX]),
        scale=np.asarray([S2_SCALE]),
        target_year_alignment="S2 temporal T24 from Jan Y-1 to Dec Y",
        nodata_value=np.asarray([S2_NODATA_VALUE], dtype=np.float32),
    )

    print("Saved final S2 temporal cache:")
    print(TRAJ_S2_TEMPORAL_NPZ)

summarize_array("X_traj_s2_temporal", X_traj_s2_temporal)
print("ids_traj_s2_temporal:", ids_traj_s2_temporal.shape)


Streaming output truncated to the last 5000 lines.
  skip rows 2430:2431 all extracted
  skip rows 2431:2432 all extracted
  skip rows 2432:2433 all extracted
  skip rows 2433:2434 all extracted
  skip rows 2434:2435 all extracted
  skip rows 2435:2436 all extracted
  skip rows 2436:2437 all extracted
  skip rows 2437:2438 all extracted
  skip rows 2438:2439 all extracted
  skip rows 2439:2440 all extracted
  skip rows 2440:2441 all extracted
  skip rows 2441:2442 all extracted
  skip rows 2442:2443 all extracted
  skip rows 2443:2444 all extracted
  skip rows 2444:2445 all extracted
  skip rows 2445:2446 all extracted
  skip rows 2446:2447 all extracted
  skip rows 2447:2448 all extracted
  skip rows 2448:2449 all extracted
  skip rows 2449:2450 all extracted
  skip rows 2450:2451 all extracted
  skip rows 2451:2452 all extracted
  skip rows 2452:2453 all extracted
  skip rows 2453:2454 all extracted
  skip rows 2454:2455 all extracted
  skip rows 2455:2456 all extracted
  skip rows 2

Assembling S2 chunks:   0%|          | 0/1357 [00:00<?, ?it/s]

Requested rows: 33354
Extracted rows: 33354
Missing rows: 0
Saved final S2 temporal cache:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/s2_temporal_T24_prev_plus_target_year/trajectory_s2_temporal_T24_prev_plus_target_year_all_zones_2017_2025.npz
X_traj_s2_temporal: shape=(33354, 24, 11), dtype=float32
  NaN: 2600560
  Inf: 0
  min/max: -13.917402267456055 32.0
ids_traj_s2_temporal: (33354,)


## 8. Final cache quality summary

In [ ]:
# ============================================================
# 8. FINAL CACHE QUALITY SUMMARY
# ============================================================

print("Final trajectory cache folder:")
print(DESTINATION_TRAJ_CACHE_DIR)

print("\nExpected trajectory rows:", len(trajectory_master_df))

# ------------------------------------------------------------
# Reload final caches
# ------------------------------------------------------------

gee_point_d = np.load(TRAJ_GEE_POINT_NPZ, allow_pickle=True)
gee_patch_d = np.load(TRAJ_GEE_PATCH_NPZ, allow_pickle=True)
s2_d = np.load(TRAJ_S2_TEMPORAL_NPZ, allow_pickle=True)

ids_gee_point = gee_point_d["traj_sample_ids"].astype(str)
ids_gee_patch = gee_patch_d["traj_sample_ids"].astype(str)
ids_s2 = s2_d["traj_sample_ids"].astype(str)

print("\nFinal cache row counts:")
print("GEE point:", len(ids_gee_point))
print("GEE patch:", len(ids_gee_patch))
print("S2 T24:   ", len(ids_s2))

# ------------------------------------------------------------
# Intersect all caches
# ------------------------------------------------------------

master_ids = set(trajectory_master_df["traj_sample_id"].astype(str))
common_ids = master_ids & set(ids_gee_point) & set(ids_gee_patch) & set(ids_s2)

print("\nRows available in all Model 5 predictor caches:", len(common_ids))
print("Rows missing at least one cache:", len(master_ids - common_ids))

if len(master_ids - common_ids) > 0:
    print("First missing IDs:")
    print(sorted(master_ids - common_ids)[:30])

# ------------------------------------------------------------
# Per-year completeness
# ------------------------------------------------------------

complete_df = trajectory_master_df[["traj_sample_id", "unit_id", "zone_id", "target_year"]].copy()
complete_df["has_gee_point"] = complete_df["traj_sample_id"].isin(ids_gee_point)
complete_df["has_gee_patch"] = complete_df["traj_sample_id"].isin(ids_gee_patch)
complete_df["has_s2"] = complete_df["traj_sample_id"].isin(ids_s2)
complete_df["has_all_model5"] = complete_df["has_gee_point"] & complete_df["has_gee_patch"] & complete_df["has_s2"]

print("\nCompleteness by target year:")
display(
    complete_df
    .groupby("target_year")
    .agg(
        n_expected=("traj_sample_id", "count"),
        n_gee_point=("has_gee_point", "sum"),
        n_gee_patch=("has_gee_patch", "sum"),
        n_s2=("has_s2", "sum"),
        n_all_model5=("has_all_model5", "sum"),
    )
    .reset_index()
)

print("\nCompleteness by zone:")
display(
    complete_df
    .groupby("zone_id")
    .agg(
        n_expected=("traj_sample_id", "count"),
        n_gee_point=("has_gee_point", "sum"),
        n_gee_patch=("has_gee_patch", "sum"),
        n_s2=("has_s2", "sum"),
        n_all_model5=("has_all_model5", "sum"),
    )
    .reset_index()
)

# ------------------------------------------------------------
# S2 quality
# ------------------------------------------------------------

s2_quality_df = pd.DataFrame({
    "traj_sample_id": s2_d["traj_sample_ids"].astype(str),
    "unit_id": s2_d["unit_ids"].astype(str),
    "zone_id": s2_d["zone_id"].astype(str),
    "target_year": s2_d["target_year"].astype(int),
    "s2_valid_months": s2_d["s2_valid_months"].astype(int),
    "s2_missing_fraction": s2_d["s2_missing_fraction"].astype(float),
    "s2_valid_obs_count_sum": s2_d["s2_valid_obs_count_sum"].astype(float),
})

print("\nS2 quality by target year:")
display(
    s2_quality_df
    .groupby("target_year")
    .agg(
        n_rows=("traj_sample_id", "count"),
        valid_months_mean=("s2_valid_months", "mean"),
        valid_months_median=("s2_valid_months", "median"),
        valid_months_min=("s2_valid_months", "min"),
        valid_months_max=("s2_valid_months", "max"),
        missing_fraction_mean=("s2_missing_fraction", "mean"),
    )
    .reset_index()
)

print("\nS2 quality by zone:")
display(
    s2_quality_df
    .groupby("zone_id")
    .agg(
        n_rows=("traj_sample_id", "count"),
        valid_months_mean=("s2_valid_months", "mean"),
        valid_months_median=("s2_valid_months", "median"),
        missing_fraction_mean=("s2_missing_fraction", "mean"),
    )
    .reset_index()
)

# ------------------------------------------------------------
# Save completeness summary
# ------------------------------------------------------------

complete_path = os.path.join(DESTINATION_TRAJ_CACHE_DIR, "model5_cache_completeness.csv")
s2_quality_path = os.path.join(DESTINATION_TRAJ_CACHE_DIR, "s2_temporal_quality_summary_rows.csv")

complete_df.to_csv(complete_path, index=False)
s2_quality_df.to_csv(s2_quality_path, index=False)

print("\nSaved quality outputs:")
print(complete_path)
print(s2_quality_path)

print("\nDone: Model 5 trajectory cache notebook complete.")


Final trajectory cache folder:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025

Expected trajectory rows: 33354

Final cache row counts:
GEE point: 33354
GEE patch: 33354
S2 T24:    33354

Rows available in all Model 5 predictor caches: 33354
Rows missing at least one cache: 0

Completeness by target year:


,target_year,n_expected,n_gee_point,n_gee_patch,n_s2,n_all_model5
0,2017,3706,3706,3706,3706,3706
1,2018,3706,3706,3706,3706,3706
2,2019,3706,3706,3706,3706,3706
3,2020,3706,3706,3706,3706,3706
4,2021,3706,3706,3706,3706,3706
5,2022,3706,3706,3706,3706,3706
6,2023,3706,3706,3706,3706,3706
7,2024,3706,3706,3706,3706,3706
8,2025,3706,3706,3706,3706,3706



Completeness by zone:


,zone_id,n_expected,n_gee_point,n_gee_patch,n_s2,n_all_model5
0,Ilebo,11862,11862,11862,11862,11862
1,Mbuji,7551,7551,7551,7551,7551
2,Mushie,11466,11466,11466,11466,11466
3,Tshikapa,2475,2475,2475,2475,2475



S2 quality by target year:


,target_year,n_rows,valid_months_mean,valid_months_median,valid_months_min,valid_months_max,missing_fraction_mean
0,2017,3706,4.010523,3.0,2,9,0.832895
1,2018,3706,3.581759,2.0,1,10,0.850760
2,2019,3706,12.395575,12.0,7,16,0.483518
3,2020,3706,21.370210,21.0,16,24,0.109575
4,2021,3706,20.927685,21.0,16,24,0.128013
5,2022,3706,20.372099,20.0,16,24,0.151163
6,2023,3706,20.504047,21.0,16,24,0.145665
7,2024,3706,20.900702,21.0,15,24,0.129137
8,2025,3706,21.765785,22.0,16,24,0.093092



S2 quality by zone:


,zone_id,n_rows,valid_months_mean,valid_months_median,missing_fraction_mean
0,Ilebo,11862,15.570899,20.0,0.351213
1,Mbuji,7551,18.000132,21.0,0.249994
2,Mushie,11466,15.572911,20.0,0.351129
3,Tshikapa,2475,16.670707,20.0,0.305387



Saved quality outputs:
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/model5_cache_completeness.csv
/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU/NN/Data/Traj_cache/all_zones_2017_2025/s2_temporal_quality_summary_rows.csv

Done: Model 5 trajectory cache notebook complete.
